# UCITS Balanced Fund

This notebook presents a UCITS risk-monitoring workflow for a simulated balanced fund. The fund combines equity, fixed income, FX, cash, and derivative exposures within a UCITS-specific regulatory framework.

The analysis focuses on UCITS monitoring outputs, including global exposure, VaR, issuer and counterparty limits, eligibility checks, stress testing and backtesting. UCITS monitoring is driven by UCITS regulatory rules and fund-specific policy settings, not by AIFMD or AIF-style risk frameworks.

> **Output gallery:** Tables and plots generated by this notebook are saved in the [fig/UCITS_Balanced](../../fig/UCITS_Balanced) folder. Readers who prefer to review the generated outputs directly can browse that folder without running the notebook.

> For supporting methodology notes, see [UCITS Balanced Notes](../../docs/notebooks_notes/ucits_balanced.md).


In [ ]:
import warnings

from src.setup_db import run as setup_db
from src.data.mock_bloomberg import MockBloomberg as Bloomberg

import src.data.database as db
import src.risk.risk_utils as rk
import src.ui.print_html_utils as phtml

warnings.filterwarnings("ignore")

setup_db()
ENGINE = db.get_engine()
BBG = Bloomberg()


### Fund Example in this Notebook

The fund profile below sets the operating context for the risk workflow. It defines the strategy, fund type, base currency, reporting setup, and monitoring framework used by the calculations that follow.

<small>

In [ ]:
# Display fund overview banner — fund identity and risk methodology framework
FUND_ID = 'UCITS_Balanced'
phtml.display_fund_overview_banner(
    fund_id=FUND_ID,
    engine=ENGINE,
    export_id="01",
)

> Note: Fund characteristics, risk limits, methodologies, and reporting parameters are simulated. They are used to show how a fund-level risk framework can be represented in a structured workflow.

---
### Risk management parameters 

The fund's monitoring parameters combine UCITS regulatory rules, here loaded as `ucits_limits` with fund-level configuration, here as `rmp`. 

**Regulatory Risk Parameters**

In [ ]:
# Load regulatory framework 
from src.data.reference_data import load_regulatory_framework

ucits_limits = load_regulatory_framework('ucits_regulatory_framework')

**Risk Management Policy**

Fund-level parameters define practical setup including benchmark selection, valuation date, reporting assumptions, and internal monitoring thresholds. The RMP is loaded as `rmp` and passed to the relevant risk functions. This keeps fund-specific parameters outside the calculation code. 

In [ ]:
# Display Risk Management Policy parameters from fund reference data
phtml.display_fund_rmp_parameters(
    fund_id=FUND_ID,
    engine=ENGINE,
    export_id="02",
)

In [ ]:
# Load RMP parameters for the fund
from src.data.reference_data import load_rmp
rmp = load_rmp(FUND_ID)


### Implementation Context

The analysis is performed as of a fixed valuation date, consistent with the point-in-time design used across the fund workflows.

In [ ]:
# fixed valuation date for all computations in this notebook
from src.config import VALUATION_DATE
VALUATION_DATE

Portfolio positions, fund characteristics, counterparty records, reference data, and market data are retrieved from the SQLite data layer. Market data is enriched through the simulated Bloomberg workflow before being passed to the risk analytics modules.

For a fuller explanation of the data workflow, see the [Data Layer Workflow](../data_workflows/01_data_layer_workflow.ipynb).

In [ ]:
# Enrich fund position data with market-data inputs
from src.data.enrichment import get_risk_ready_df
risk_df = get_risk_ready_df(ENGINE, FUND_ID, VALUATION_DATE)

# display the enriched risk dataframe first 2 rows
risk_df.head(2)


From this point onward, the notebook uses helper functions to render summary tables and HTML views. Data aggregation, calculations, and reporting logic remain inside the package modules, so the notebook stays focused on review and interpretation.

## 1. Load Positions

Load today's UCITS Balanced Fund positions from the daily fund
administrator export.

In [ ]:
# Query positions from SQLite
from src.data.database import query_positions
positions = query_positions(ENGINE, FUND_ID, VALUATION_DATE)

# Enrich positions with market-data inputs
risk_df = get_risk_ready_df(ENGINE, FUND_ID, VALUATION_DATE)

# info in risk_df covers many risk inputs
NAV = risk_df['market_value_eur'].sum()


## 2. Position Data and Portfolio Overview
UCITS imposes strict eligibility and concentration rules that must be checked at the position level before any risk calculation begins.

In [ ]:
phtml.display_fund_summary(FUND_ID, VALUATION_DATE, positions, risk_df, NAV)

In [ ]:
phtml.display_top_positions(risk_df, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="03")

In [ ]:
phtml.display_asset_class_breakdown(risk_df, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="04")  # Also computes NAV internally

## 3. UCITS Eligibility and Investment Restriction Check


In [ ]:
# UCITS position-level compliance checks
from src.risk.ucits_compliance_checks import run_ucits_compliance_checks

compliance_result = run_ucits_compliance_checks(positions, NAV)
phtml.display_ucits_compliance_checks(compliance_result, export_id="02a", fund_id=FUND_ID, valuation_date=VALUATION_DATE)

## 4. UCITS Global Exposure Framework

UCITS global exposure is monitored through one selected method: commitment, absolute VaR, or relative VaR.

The selected method defines the active control:
- commitment: 100% TNA limit
- absolute VaR: 20% TNA limit
- relative VaR: 200% of reference portfolio VaR

Derivative notional leverage, EPM leverage, and borrowing are monitored separately.

In [ ]:
# global exposure params to use later
global_exposure = rmp["global_exposure_policy"]

## 5. VaR and Expected Shortfall

The fund uses absolute VaR for UCITS global exposure monitoring. Model choices are taken from the fund's RMP, and the regulatory absolute VaR limit is taken from the central UCITS framework.

Expected Shortfall is shown as a complementary market risk measure.


In [ ]:
# get regulatory parameters related to var
ucits_var = ucits_limits['var_framework']

var_confidence = ucits_var['confidence_level']
var_holding_period = ucits_var['holding_period_days']
var_lookback = ucits_var['lookback_window_days']
absolute_var_limit_pct = ucits_var['absolute_limit_pct']
relative_var_limit = ucits_var['relative_limit_multiplier']

# Get reference portfolio from RMP's global exposure policy
relative_var_reference = rmp['global_exposure_policy']['reference_portfolio_id']

# Unpack risk parameters related to var from RMP
rmp_var = rmp['var_framework']
df = rmp_var['parametric_degrees_of_freedom']

In [ ]:
# 1-day fixed-position VaR + ES computation (1 day and scaled)
from src.pipeline.fixed_position_var import compute_fixed_position_var_1day

var_result = compute_fixed_position_var_1day(
    engine=ENGINE,
    fund_id=FUND_ID,
    valuation_date=VALUATION_DATE,
    confidence=var_confidence,
    df=df,
    horizon=var_holding_period,
)

# Display VaR and ES (extracts nav, horizon, and valuation_date from var_result)
phtml.display_var_es(var_result, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="05")

## 6. Relative VaR

Relative VaR is shown as an additional reference portfolio comparison. The selected UCITS global exposure method for this fund is absolute VaR.

$$\text{Relative VaR} = \frac{VaR_{fund}}{VaR_{reference}} \times 100$$

The reference portfolio for this UCITS Balanced fund is a 60/40 allocation, with 60% MSCI World and 40% EUR government bonds, as documented in the fund's risk management policy.


In [ ]:
from src.risk.ucits_relative_var import compute_ucits_relative_var_point_in_time
rel_var_result = compute_ucits_relative_var_point_in_time(
    ENGINE, FUND_ID, var_result, var_confidence, var_lookback, var_holding_period, relative_var_limit, BBG, VALUATION_DATE, rmp
)

phtml.display_ucits_relative_var_point_in_time(
    rel_var_result,
    valuation_date=VALUATION_DATE,
    fund_id=FUND_ID,
    export_id="06"
)

## 7. VaR Backtesting and Monitoring

VaR backtesting compares the VaR estimate with reconstructed historical P&L for the portfolio being measured. The portfolio is held fixed, and historical market moves are applied to test how often losses would have exceeded the VaR estimate.

The fund uses two checks for VaR model monitoring, breach review, and internal model validation:

- **Kupiec POF**: tests whether the breach rate is consistent with the expected rate for the VaR confidence level.
- **Christoffersen**: tests whether VaR breaches appear independently distributed.


In [ ]:
# VaR Backtest — rolling 1-day VaR over 250 trading days (changeable)
from src.risk.var_backtest import compute_var_backtest_rolling, create_backtest_report
import pandas as pd

# Compute rolling VaR backtest
backtest_lookback = rmp['backtesting']['lookback_days']
start_date = (pd.Timestamp(VALUATION_DATE) - pd.tseries.offsets.BDay(backtest_lookback)).strftime('%Y-%m-%d')

backtest_df = compute_var_backtest_rolling(
    engine=ENGINE,
    fund_id=FUND_ID,
    start_date=start_date,
    end_date=VALUATION_DATE,
    lookback=backtest_lookback,
)

# Create report with Kupiec & Christoffersen tests (95%, 97.5%, 99%)
report = create_backtest_report(backtest_df)

# Display using HTML table
phtml.display_backtest_report(report, window_size=backtest_lookback, valuation_date=VALUATION_DATE, model="Historical", fund_id=FUND_ID, export_id="07")

In [ ]:
from src.ui.backtest_plot import plot_var_backtest

# Extract properly aligned returns and VaR for plotting
# So we use returns[1:] and vars[:-1]
returns_shifted = backtest_df['realised_return'].iloc[1:].values
var_shifted = backtest_df['var_99_pct'].iloc[:-1].values
dates_shifted = backtest_df['date'].iloc[1:].values

# Extract Kupiec and Christoffersen p-values from report 
kupiec_pvalue = report[report['confidence'] == var_confidence*100]['kupiec_p'].values[0] if len(report[report['confidence'] == var_confidence*100]) > 0 else None
chris_pvalue = report[report['confidence'] == var_confidence*100]['christoffersen_p'].values[0] if len(report[report['confidence'] == var_confidence*100]) > 0 else None

fig, ax = plot_var_backtest(
    dates_shifted,
    returns_shifted,
    var_shifted,
    FUND_ID,
    title=f'VaR Backtest — {FUND_ID} (99% confidence, 250-day lookback)',
    kupiec_pvalue=kupiec_pvalue,
    christoffersen_pvalue=chris_pvalue,
    valuation_date=VALUATION_DATE,
    export_id="08"
)

**Absolute and Relative VaR Monitoring Report**

## 8. SRRI and KIID Monitoring

This section calculates the fund's SRRI and monitors whether changes in the indicator would require attention for investor disclosure purposes.

### 8.1 SRRI Methodology and Current Indicator

The SRRI is computed from 5 years of weekly NAV returns, annualised, and mapped to a 1-7 category under CESR/10-673.

CESR/10-673 defines the methodology used for the SRRI shown in the UCITS KIID, including the weekly return window, annualisation approach, and volatility buckets used to assign the indicator.

| SRRI | Volatility range |
| ---- | ---------------- |
| 1    | < 0.5%           |
| 2    | 0.5% - 2%        |
| 3    | 2% - 5%          |
| 4    | 5% - 10%         |
| 5    | 10% - 15%        |
| 6    | 15% - 25%        |
| 7    | >= 25%           |



In [ ]:
from src.risk.ucits_srri import compute_srri_from_fund

srri_result = compute_srri_from_fund(ENGINE, FUND_ID)

phtml.display_ucits_srri_point_in_time(
    srri_result,
    valuation_date=VALUATION_DATE,
    fund_id=FUND_ID,
    export_id="09"
)

srri = srri_result['sri_bucket']

### 8.2 SRRI Monitoring and KIID Update Trigger

The notebook monitors SRRI using a rolling weekly return window and checks whether a category change persists long enough to require a KIID update review.

The displayed history shows the recent SRRI category, annualised volatility, and update flag. Observation counts are kept as internal validation fields rather than shown in the report table.


In [ ]:
# === CANONICAL: Section 9 — Rolling SRRI and KIID Update Trigger ===
from src.risk.ucits_srri import compute_srri_rolling_monthly

# Compute rolling monthly SRRI with 4-month persistence check
srri_config = rmp['srri_monitoring']
srri_rolling = compute_srri_rolling_monthly(
    engine=ENGINE,
    fund_id=FUND_ID,
    as_of_date=VALUATION_DATE,
    current_disclosed_srri=srri_config['current_disclosed_srri'],
    window_weeks=srri_config.get('window_weeks', 260),
    persistence_months=srri_config.get('category_change_persistence_months', 4),
)

# Display rolling SRRI monitoring with volatility, SRRI trajectory, and KIID triggers
current_disclosed_srri = srri_config['current_disclosed_srri']
phtml.display_srri_monitoring(
    srri_rolling,
    current_disclosed_srri=current_disclosed_srri,
    fund_id=FUND_ID,
    valuation_date=VALUATION_DATE,
    export_id="10",
)

In [ ]:
from src.risk.ucits_var_monitoring import compute_var_monitoring_summary

summary_df = compute_var_monitoring_summary(
    var_result, rel_var_result, srri_result, report,
    absolute_var_limit_pct, relative_var_limit
)

phtml.display_ucits_var_monitoring_summary(
    summary_df,
    valuation_date=VALUATION_DATE,
    fund_id=FUND_ID,
    export_id="11"
)

## 9. Stress Testing

This section applies the CSSF UCITS stress tests.

The univariate stress tests are the prescribed CSSF shocks. The most relevant historical scenarios are selected for the fund based on its risk profile and risk characteristics.

The stress results use a simplified first-order approach, not full revaluation.


In [ ]:
# UCITS Stress Testing — CSSF Prescribed Scenarios
# Load scenarios from central configuration

from src.risk.ucits_stress_scenarios import compute_ucits_stress_scenarios

# Compute all CSSF prescribed univariate and selected historical scenarios
scenarios_result = compute_ucits_stress_scenarios(FUND_ID, risk_df)

phtml.display_ucits_scenarios(
    risk_df,
    scenarios_result=scenarios_result,
    valuation_date=VALUATION_DATE,
    fund_id=FUND_ID,
    export_id="12"
)

## 10. Liquidity, Redemption, and Investor Profile

This section assesses whether the fund’s assets are aligned with its redemption terms. It covers asset liquidity buckets, redemption stress, and investor concentration.

For a daily-dealing UCITS, investor concentration is considered as part of liquidity monitoring because large subscriptions or redemptions may affect portfolio trading, cash-buffer usage, or escalation.

The investor register is simulated. The output should be read as a liquidity governance indicator, not as an AIFMD Annex IV investor-concentration report.


### 10.1 Buckets

The liquidity profile classifies portfolio assets by estimated time to liquidation under normal market conditions.

<small>

| Bucket | Time to liquidate |
| ------ | ----------------- |
| 1      | 1 day             |
| 2      | 2-7 days          |
| 3      | 8-30 days         |
| 4      | 31-90 days        |
| 5      | 91-365 days       |
| 6      | > 1 year          |

</small>

Days to liquidate are estimated per position as:

$$
\text{days}_i = \frac{|MV_i|}{ADV_i \times pct\_adv}
$$

* **ADV** is the average daily volume sourced from the market-data layer.
* **pct_adv** is an RMP parameter defining the maximum share of ADV assumed tradeable per day.
* Cash and money-market instruments are classified as 1 day.
* Instruments with no reliable trading volume are treated as illiquid.


In [ ]:
# Risk parameters from RMP
liquidity_pct_adv = rmp['liquidity_monitoring']['pct_adv']

In [ ]:
# Liquidity profile
liq = rk.compute_liquidity_profile(risk_df, pct_adv=liquidity_pct_adv)
risk_df_liq = liq['risk_df_liq']
bucket_full = liq['bucket_full']

phtml.display_buckets(
    bucket_full, risk_df_liq, NAV, 
    valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="13")

In [ ]:
from src.ui.liquidity_plot import plot_liquidity_profile

x = plot_liquidity_profile(
    bucket_full, FUND_ID, metric='pct_nav_abs', 
    valuation_date=VALUATION_DATE, export_id="14")

---

### 10.2 Redemption Stress

Redemption stress compares assets liquidatable within the fund's notice period against potential redemption requests. 


### 10.3 Investor Concentration

Investor concentration is monitored as a fund-level governance indicator.

The largest-investor scenario connects the investor register to redemption stress by translating investor concentration into a liquidity demand.

In [ ]:
# Liquidity profile and redemption stress
from src.pipeline.liquidity_policy import load_redemption_scenarios_from_rmp

liq = rk.compute_liquidity_profile(risk_df, pct_adv=liquidity_pct_adv)
risk_df_liq = liq['risk_df_liq']

# Redemption stress scenarios — load from RMP
notice_period_days = rmp['notice_period_days']
REDEMPTION_SCENARIOS = load_redemption_scenarios_from_rmp(FUND_ID)

phtml.display_redemption_stress(
    FUND_ID,
    notice_period_days,
    REDEMPTION_SCENARIOS,
    NAV,
    risk_df_liq,
    valuation_date=VALUATION_DATE,
    export_id="15",
)

## 11. Counterparty and Collateral Risk

UCITS Article 52 limits counterparty risk exposure from non-centrally-cleared derivative transactions to 10% of fund assets where the counterparty is an eligible credit institution, and 5% in other cases.

For this workflow, counterparty exposure is represented as the positive mark-to-market exposure after legally enforceable netting and eligible collateral received by the fund. 

> Collateral and margin balances are simulated. A production workflow would source these figures from daily counterparty exposure, collateral, and margin reconciliation data.

In [ ]:
# Counterparty risk — UCITS
cp_result = rk.compute_counterparty_stress(FUND_ID, ENGINE, NAV)

phtml.display_counterparty_risk_ucits(
    NAV,
    cp_result,
    fund_id=FUND_ID,
    valuation_date=VALUATION_DATE
)

---

## 12. Pre-Trade Compliance Checks

For UCITS, pre-trade checks are used to test whether proposed trades remain within eligibility, concentration, borrowing and exposure constraints under the UCITS framework.

`pre_trade_check` runs the following checks:
- **Absolute VaR** post-trade < 20% NAV (20-day, 99%)
- **Relative VaR** < 2× reference portfolio
- **5/10/40 rule** (Article 52): single issuer < 10% NAV; issuers > 5% aggregate to < 40%
- **Eligible assets** (Article 50): no direct real estate, loans, CLOs, private equity
- **Counterparty exposure** (OTC derivatives): < 10% for credit institutions, < 5% otherwise
- **Borrowing limit** < 10% NAV (Article 83)

> **ETF look-through limitation.** This portfolio is dominated by broad index ETFs (SPY 48%, Euro Stoxx 28%). In a fully compliant implementation, these ETFs would be evaluated on a look-through basis. The simplified `pre_trade_check` applies the 5/10/40 rule at ETF level. The relevant compliance question for each scenario below is whether the *proposed trade* introduces a new or worsened breach relative to the current portfolio state.


In [ ]:
# Import removed — use rk.pre_trade_check instead (available from setup cell)

### 12.1 Scenario 1 - Small IG bond buy

Buy BMW AG bonds, new issuer not currently in the portfolio. \
EUR 5M notional on a EUR 515M NAV = 0.97% of NAV. \
The trade is small and compliant. The check will surface the pre-existing SPY and \
Euro Stoxx concentrations (ETF look-through limitation), but the proposed trade does not \
introduce any new or worsened breach — which is the material compliance question.\


In [ ]:
# BMW AG IG bond — new issuer, small position, eligible instrument
trade_bond = {
    'isin'           : 'XS2600000001',
    'direction'      : 'buy',
    'quantity'       : 5_000_000,         # EUR 5 M at par
    'price_eur'      : 1.00,
    'asset_class'    : 'Bond',
    'sub_asset_class': 'IG Corporate',
    'dur_adj_mid'    : 3.5,
    'currency'       : 'EUR',
    'adv_eur'        : 10_000_000,
    'issuer'         : 'BMW AG',
    'sector'         : 'Automobiles',}



result_conc = rk.pre_trade_check(trade_bond, ENGINE, FUND_ID, VALUATION_DATE)
phtml.display_ptc(result_conc)

### 12.2 Scenario 2 - Single-stock concentration breach (5/10/40 rule, Article 52)

Buy Apple Inc (not in the UCITS portfolio), 350,000 shares at EUR 172/share \
= EUR 60.2M notional. On NAV of EUR 514.8M, this is **11.7%** of NAV — \
a clear breach of the 10% single-issuer hard limit under UCITS Directive Article 52. \
The check flags the trade as non-compliant with the single-issuer limit.

The 10% limit applies per *issuer* of transferable securities, not per instrument. \
If the same issuer were held in both equity and bond form, both exposures would be \
aggregated for the 5/10/40 test. A UCITS portfolio can hold up to six issuers at 5–10% \
(the "40 rule" caps the aggregate of all positions > 5%), but no single issuer may exceed 10%.


In [ ]:
# Apple Inc — new single-stock position sized to breach 10% issuer limit (11.7% NAV)
trade_conc = {
    'isin'           : 'US0231351067',
    'direction'      : 'buy',
    'quantity'       : 350_000,           # 350 k × EUR 172 = EUR 60.2 M
    'price_eur'      : 172.00,
    'asset_class'    : 'Equity',
    'sub_asset_class': 'Large Cap',
    'beta'           : 1.25,
    'currency'       : 'USD',
    'adv_eur'        : 200_000_000,
    'issuer'         : 'Apple Inc',
    'sector'         : 'Information Technology',
}

result_conc = rk.pre_trade_check(trade_conc, ENGINE, FUND_ID, VALUATION_DATE)
phtml.display_ptc(result_conc)

### 12.3 Scenario 3 - Ineligible asset (UCITS Directive Article 50)

The portfolio manager proposes to invest EUR 10M in a direct real estate holding. \
Direct property is not a transferable security, money market instrument, unit in a UCITS/UCI, \
bank deposit, or eligible derivative — it is therefore ineligible under UCITS Directive Article 50. \
The check flags the trade as ineligible under the UCITS asset eligibility rules. \
The check treats this as an eligibility breach rather than an internal risk-limit breach.


In [ ]:
# Direct property — ineligible under UCITSD Art. 50
trade_inelig = {
    'isin'           : 'LU-PROP-DIRECT-001',
    'direction'      : 'buy',
    'quantity'       : 1,
    'price_eur'      : 10_000_000,
    'asset_class'    : 'Real Estate',
    'sub_asset_class': 'Direct Property',
    'currency'       : 'EUR',
    'adv_eur'        : 0,
    'issuer'         : 'Luxembourg Office Holdings SA',
    'sector'         : 'Real Estate',
}

result_inelig = rk.pre_trade_check(trade_inelig, ENGINE, FUND_ID, VALUATION_DATE)
phtml.display_ptc(result_inelig)

## 13. P&L Attribution and Model Review
P&L attribution explains daily portfolio return drivers by risk factor. In this notebook, it is used as an internal model review and governance output for the UCITS risk workflow.

The value bridge methodology decomposes daily P&L into:

- market beta
- rates
- FX
- residual or unexplained P&L

Benchmark: SX5E (Euro Stoxx 50). The benchmark is used as the equity-market factor in the attribution model for this simulated EUR-denominated balanced fund.

The residual captures P&L not explained by the selected market, rates, and FX factors.

> Sensitivity-based attribution uses daily positions and daily market moves, consistent with the position-based risk workflow used in the notebook.


In [ ]:
from src.risk.pnl_attribution import compute_daily_attribution
from src.ui.attribution_plot import plot_attribution_cumsum

# Compute attribution with 60/40 balanced benchmark
attr_result = compute_daily_attribution(
    engine=ENGINE,
    fund_id=FUND_ID,
    bbg=BBG,
    valuation_date=VALUATION_DATE,
    residual_threshold_pct=0.20,
    benchmark='60/40',
)

attr = attr_result['attr']
flagged = attr_result['flagged']
attr_cumsum = attr_result['attr_cumsum']

# Plot cumulative attribution
plot_attribution_cumsum(attr_cumsum, FUND_ID, valuation_date=VALUATION_DATE, export_id="16")

# Model quality report
import src.print_utils as prt
prt.print_attribution(attr, flagged)

**Methodology and limitations**

Sensitivity-based attribution using actual daily positions and market moves.
Regression-based attribution is not used — it gives average historical loadings
and cannot reflect current position composition.

Benchmark: SX5E (Euro Stoxx 50) — appropriate for this EUR-denominated balanced fund.

**Limitations:**
* Single-factor equity model — no sector, style, or stock-level decomposition
* Rate attribution uses a simulated parallel shift — no key-rate DV01
* FX covers EUR/USD only
* Position composition is static in this simulation

**Regulatory context:** this section is an internal governance output consumed
by the Board and risk committee. It supports the AIFMD Article 15 evidence
pack and CSSF expectations for risk management reporting. It is not a direct
Annex IV or Annex VI deliverable.

## 14. ESG / Sustainability Indicators
The notebook computes portfolio-level ESG indicators using NAV-weighted averages and the fund's internal ESG threshold.

Metrics monitored include weighted average ESG score, percentage of NAV below the internal threshold, controversy flags, and carbon intensity.

ESG scores for liquid instruments are sourced from Bloomberg. Instruments without a Bloomberg ticker use fund administrator ESG data.

ESG scores should be treated as sustainability-risk or disclosure indicators unless explicitly mapped to SFDR Article 6, 8, or 9 concepts.


In [ ]:
import src.risk.esg_utils as esg_u 
esg_df  = esg_u.build_esg_df(risk_df, BBG, ENGINE, FUND_ID, VALUATION_DATE)
esg_u.display_esg_assets(esg_df, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="22")

In [ ]:
esg_u.display_esg_summary(esg_df, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="23")

In [ ]:
esg_u.plot_esg_profile(esg_df, FUND_ID, plot_title='22. ESG profile - UCITS', valuation_date=VALUATION_DATE, export_id="24")